# base_rh.ipynb — Semana 04 · Dia 01
## Leitura de arquivos + módulo datetime
**SENAI · Visualização de Dados e BI · Turma T2**

---

### Como usar este notebook
- Execute cada célula na ordem, uma por vez (**Command + Enter**)
- Leia os comentários antes de rodar — eles explicam o que acontece
- Se uma célula der erro, leia a mensagem: ela diz exatamente o que faltou
- Ao final, todo o código deste notebook vira o seu `base_rh.py`

---

### O que você vai aprender hoje
1. Como importar bibliotecas no Python
2. Como ler um CSV com pandas — e por que os parâmetros importam
3. Os comandos básicos de exploração de um DataFrame
4. Como trabalhar com datas no Python
5. Como exportar dados para JSON e Excel
6. Como criar funções para reutilizar código

---

> 🏭 **Contexto real:** você foi contratado como analista de dados júnior em uma indústria de Jaraguá do Sul. O time de RH mandou a base de funcionários. Antes de qualquer análise, você precisa entender o que veio.

---
## 📦 CÉLULA 1 — Importando as bibliotecas

**O que é uma biblioteca?**  
É um conjunto de ferramentas prontas que alguém já escreveu para você usar.  
Você não precisa criar tudo do zero — só importar e usar.

| Biblioteca | Para que serve                                  | Como importar                   |
| ------------| -------------------------------------------------| ---------------------------------|
| `pandas`   | Ler, transformar e analisar dados em tabelas    | `import pandas as pd`           |
| `re`       | Encontrar e substituir padrões em texto (regex) | `import re`                     |
| `datetime` | Trabalhar com datas e horas                     | `from datetime import datetime` |

**Por que `as pd`?**  
É um apelido. Em vez de escrever `pandas.read_csv()` toda vez, você escreve `pd.read_csv()`.  
É uma convenção do mercado — todo analista usa `pd` para pandas.

In [ ]:
# Sempre a primeira célula do notebook: importar o que você vai usar
# Se aparecer erro aqui, a biblioteca não está instalada.
# Solução: abra o terminal e rode:  pip install pandas openpyxl

import pandas as pd      # biblioteca de análise de dados
import re                # expressões regulares (para limpar texto)
from datetime import datetime  # para trabalhar com datas

# Confirmar que as importações funcionaram
print('✓ pandas versão:', pd.__version__)
print('✓ Bibliotecas importadas com sucesso!')

---
## 📂 CÉLULA 2 — Lendo o arquivo CSV

**O que é um CSV?**  
É um arquivo de texto onde os dados ficam separados por um caractere (vírgula, ponto-e-vírgula etc.)  
CSV = *Comma-Separated Values* = Valores Separados por Vírgula

**Por que precisamos de parâmetros especiais?**  
O arquivo `base_rh.csv` foi gerado pelo **TOTVS** — sistema ERP muito usado em indústrias da região.  
O TOTVS salva arquivos no padrão europeu/Windows, que é diferente do padrão internacional:

| Parâmetro | Por que usar | O que acontece se não usar |
|---|---|---|
| `sep=";"` | O TOTVS separa colunas com `;` | O pandas lê tudo como 1 coluna só |
| `encoding="cp1252"` | Encoding do Windows — suporta ã, é, ç | "Produção" vira "Produ??o" |
| `decimal=","` | No Brasil o decimal é vírgula: 9.088,34 | Salário vira texto, não número |

In [ ]:
# pd.read_csv()     para ler arquivos csv
# pd.read_excel()   para ler arquivos excel
# pd.read_json()    para ler arquivos json
# pd.read_sql()     para ler dados de um banco de dados SQL
# pd.read_html()    para ler tabelas de uma página web
# pd.read_clipboard() para ler dados copiados para a área de transferência

# CAMINHO DO ARQUIVO CSV - pode ser um caminho absoluto ou relativo
# caminho_absoluto = "/Users/samuelbucco/Documents/SCTec/Análise de Dados/Etapa Profissionalizar/Mod 1 - Modelagem de Dados/turma-visualizacao-de-dados/alunos/samuel_bucco/semana_04/base_rh.csv"
# caminho_relativo = "base_rh.csv"

caminho = "base_rh.csv"  # usando caminho relativo, assumindo que o arquivo está na mesma pasta do notebook
df = pd.read_csv(
    caminho,
    sep = ";",  # separador de campos, no caso ponto e vírgula
    encoding = "cp1252",  # codificação do arquivo, pode ser utf-8, latin-1, etc.
    decimal=','  # separador decimal, no caso vírgula
    )

print("dados carregados com sucesso!")
print(f"o dataframe tem {df.shape[0]} linhas e {df.shape[1]} colunas.")

**O que é `df.shape`?**  
`shape` é uma propriedade do DataFrame que retorna uma tupla `(linhas, colunas)`.  
- `df.shape[0]` → número de linhas (registros)
- `df.shape[1]` → número de colunas (campos)

Você vai usar isso em **todo projeto** para confirmar que leu o arquivo certo.

---
## 🔍 CÉLULA 3 — Primeiras olhadas no dado

Antes de mexer em qualquer coisa, você precisa **ver** o que chegou.  
Esses três comandos são os mais usados no dia a dia de um analista.

In [ ]:
# df.head(n) → mostra as PRIMEIRAS n linhas do DataFrame
# Padrão: head(5) se você não passar nenhum número
# Use para ter uma rápida visão geral do conteúdo

df.head(5)

In [ ]:
# df.tail(n) → mostra as ÚLTIMAS n linhas
# Serve para confirmar que o arquivo foi lido até o fim
# Se o arquivo tiver 1.000 linhas, o tail deve mostrar IDs próximos de 1000

df.tail(3)

In [ ]:
# df.columns → lista com o nome de todas as colunas
# .tolist() converte o resultado para uma lista Python comum

print('Colunas do dataset:')
print(df.columns.tolist())

---
## 🩺 CÉLULA 4 — Diagnóstico: tipos e nulos

**O que são tipos de dados?**  
Cada coluna armazena um tipo de informação. O Python precisa saber o tipo  
para saber o que pode fazer com ela:

| Tipo | O que é | Exemplo |
|---|---|---|
| `int64` | Número inteiro | 1, 42, 1000 |
| `float64` | Número decimal | 9088.34, 3.14 |
| `object` | Texto / string | "Julia Nunes", "Ativo" |
| `datetime64` | Data e hora | 2024-08-13 |

**O que são nulos?**  
São células vazias — o campo não foi preenchido no sistema de origem.  
Em Python/pandas, nulos aparecem como `NaN` (*Not a Number*) ou `NaT` (*Not a Time*).

In [ ]:
# df.info() → resumo completo do DataFrame em uma só chamada:
# - Nome de cada coluna
# - Quantidade de valores não-nulos
# - Tipo de dado (dtype)
# - Uso de memória
# É o comando mais importante para o primeiro diagnóstico!

df.info()

In [ ]:
# df.isnull() → cria uma tabela de True/False (True = é nulo)
# .sum()      → soma os True de cada coluna (True vale 1, False vale 0)
# Resultado: quantidade de nulos por coluna

print('Quantidade de nulos por coluna:')
print(df.isnull().sum())

print()

# Percentual de nulos — mais útil em bases grandes
# len(df) = número total de linhas
print('Percentual de nulos por coluna:')
print((df.isnull().sum() / len(df) * 100).round(2))

---
## 📊 CÉLULA 5 — Estatísticas das colunas numéricas

`describe()` gera um resumo estatístico automático.  
Você vai usar isso para detectar valores suspeitos (ex: salário de R$0,00 ou idade de 200 anos).

| Estatística | O que significa |
|---|---|
| `count` | Quantos valores não-nulos existem |
| `mean` | Média |
| `std` | Desvio padrão — o quanto os valores variam em torno da média |
| `min` | Menor valor |
| `25%` | 25% dos registros estão abaixo desse valor |
| `50%` | Metade dos registros estão abaixo (mediana) |
| `75%` | 75% dos registros estão abaixo desse valor |
| `max` | Maior valor |

In [ ]:
# df.describe() → estatísticas das colunas numéricas
# .round(2)    → arredonda para 2 casas decimais

df.describe().round(2)

---
## 🗂️ CÉLULA 6 — Explorando colunas categóricas

**Coluna categórica** = coluna com poucos valores possíveis (departamento, cargo, status...)  

`nunique()` conta quantos valores diferentes existem.  
`value_counts()` mostra a distribuição — quantas vezes cada valor aparece.

In [ ]:
# df.nunique() → retorna quantos valores ÚNICOS cada coluna tem
# Serve para identificar quais são categóricas (poucas opções)
# e quais são identificadoras (muitas opções diferentes)

print('Valores únicos por coluna:')
print(df.nunique())

In [ ]:
# df['coluna'].value_counts() → conta quantas vezes cada valor aparece,
# ordenado do mais frequente para o menos frequente

# Vamos ver todas as colunas categóricas de uma vez usando um loop:
# 'for col in lista' = para cada coluna na lista, executa o bloco abaixo

for col in ['Departamento', 'Cargo', 'Genero', 'Status', 'Estado_Civil']:
    print(f'\n── {col} ──')   # f-string: {} insere o valor da variável
    print(df[col].value_counts())

---
## 🔎 CÉLULA 7 — Selecionando colunas e filtrando linhas

São as operações mais usadas no dia a dia. Você vai usar isso em todo projeto.

**Seleção de coluna** → `df['Nome']`  
**Seleção de múltiplas colunas** → `df[['Nome', 'Salario']]`  
**Filtro de linhas** → `df[df['Status'] == 'Ativo']`

In [ ]:
# Selecionar UMA coluna → retorna uma Series (lista com índice)
# Uma Series é como uma coluna de planilha isolada

df['Nome'].head(5)

In [ ]:
# Selecionar MÚLTIPLAS colunas → retorna um DataFrame
# Note os dois colchetes: df[  ['col1', 'col2']  ]
# O primeiro [] é o operador de acesso do DataFrame
# O segundo [] é a lista de colunas

df[['Nome', 'Departamento', 'Cargo', 'Salario']].head(6)

In [ ]:
# FILTRO SIMPLES: selecionar só as linhas onde Status == 'Ativo'
# df['Status'] == 'Ativo' → cria uma Series de True/False
# df[ True/False ]        → mantém só as linhas onde é True

ativos = df[df['Status'] == 'Ativo']

print(f'Total de funcionários : {len(df)}')
print(f'Funcionários ativos   : {len(ativos)}')
print(f'Funcionários inativos : {len(df) - len(ativos)}')

In [ ]:
# FILTRO COMPOSTO: duas condições ao mesmo tempo
# Use & para AND (as duas condições precisam ser verdadeiras)
# Use | para OR  (pelo menos uma precisa ser verdadeira)
# IMPORTANTE: cada condição precisa estar entre parênteses

gerentes_ti = df[
    (df['Cargo'] == 'Gerente') & (df['Departamento'] == 'TI')
]

print(f'Gerentes na área de TI: {len(gerentes_ti)}')
print()
gerentes_ti[['Nome', 'Salario', 'Status', 'Data_Admissao']]

In [ ]:
# Estatísticas rápidas de uma coluna numérica
# .mean()  → média
# .max()   → maior valor
# .min()   → menor valor
# .median()→ mediana (valor do meio quando ordenados)

print(f"Salário médio : R$ {df['Salario'].mean():,.2f}")
print(f"Salário máximo: R$ {df['Salario'].max():,.2f}")
print(f"Salário mínimo: R$ {df['Salario'].min():,.2f}")
print(f"Salário mediana:R$ {df['Salario'].median():,.2f}")

# :,.2f → formata o número com separador de milhar (,) e 2 casas decimais (.2f)

---
## 📁 CÉLULA 8 — Exportando para JSON

**O que é JSON?**  
É o formato padrão de troca de dados entre sistemas modernos — APIs, ERPs na nuvem, dashboards web.  
JSON = *JavaScript Object Notation*

```json
[
  { "Nome": "Julia Nunes", "Departamento": "Logística", "Salario": 9088.34 },
  { "Nome": "Gustavo Duarte", "Departamento": "TI", "Salario": 8155.98 }
]
```

**Por que `orient="records"`?**  
Define o formato do JSON. `records` = cada linha vira um objeto separado.  
É o formato que APIs REST esperam receber.